# Laboratorio 24. Detección de anomalías con autoencoders: transacciones atípicas

**Pregunta guía:** ¿Puede un *autoencoder* entrenado sin etiquetas priorizar transacciones sospechosas mejor que reglas univariadas y que la distancia de Mahalanobis, dada una capacidad limitada de revisión?

## Objetivos
Al finalizar podrás construir variables de transacciones y estandarizarlas con train; implementar dos baselines (z-score máximo y Mahalanobis); entrenar un *autoencoder* y usar el error de reconstrucción como puntaje de anomalía; evaluar con precisión promedio y *recall* según la capacidad de revisión, por tipo de anomalía; y explicar una alerta por variable.

## Prerrequisitos y conexión
Usa redes densas y entrenamiento (laboratorios 04 a 08) y *early stopping* (laboratorio 20). Retoma la pregunta del laboratorio 23: qué hacer con casos que no tienen etiqueta. Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Un *autoencoder* comprime la entrada en un código de baja dimensión y la reconstruye: $\hat{x}=g_\phi(f_\theta(x))$, minimizando $\|x-\hat{x}\|^2$. Si se entrena con datos mayoritariamente normales, aprende la estructura del comportamiento habitual, incluidas relaciones **entre** variables. Una transacción que no respeta esa estructura se reconstruye mal: su error $e(x)=\frac{1}{p}\sum_j(x_j-\hat{x}_j)^2$ es el puntaje de anomalía. Mahalanobis también usa relaciones entre variables, pero solo lineales: $d^2=(x-\mu)^\top\Sigma^{-1}(x-\mu)$.

## Problema y datos
Un emisor de tarjetas ficticio procesa 30 000 transacciones con categoría de comercio, monto, hora, canal (presencial u online), distancia al domicilio y número de transacciones en las últimas 24 horas. Insertamos un 1,2 % de anomalías de tres tipos: **monto extremo de madrugada**, **ráfaga** de transacciones y **combinación atípica** (compra online registrada lejos del domicilio en una categoría casi siempre presencial). En la combinación atípica cada valor por separado es habitual; lo raro es la combinación. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
El equipo de fraude puede revisar un número fijo de alertas por día; el modelo debe **ordenar** bien, no clasificar perfecto. Las etiquetas llegan tarde e incompletas (solo se confirma lo que se revisa), por eso los métodos no supervisados son útiles. Las alertas deben poder explicarse al cliente y al regulador.

## Experimento conceptual
¿Qué tipos de anomalía detecta cada método dentro de la capacidad de revisión? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar transacciones normales y anómalas
El comportamiento normal tiene estructura: cada categoría tiene su monto típico y su probabilidad de compra online, los restaurantes se concentran en la noche y las compras online no registran distancia física.

In [ ]:
n_tx = 30000
category_names = ['supermercado', 'combustible', 'restaurantes', 'tecnologia',
                  'viajes', 'farmacia', 'vestuario', 'servicios']
category_probs = np.array([0.28, 0.12, 0.15, 0.06, 0.04, 0.12, 0.10, 0.13])
mean_log_amount = np.array([10.0, 10.3, 9.8, 11.5, 12.0, 9.5, 10.5, 10.2])  # log de CLP
online_prob = np.array([0.15, 0.02, 0.10, 0.60, 0.70, 0.10, 0.40, 0.80])

category = rng.choice(8, n_tx, p=category_probs)
log_amount = rng.normal(mean_log_amount[category], 0.5)
hour = rng.normal(np.where(category == 2, 20.5, 14.0), np.where(category == 2, 2.0, 3.5)) % 24
online = (rng.random(n_tx) < online_prob[category]).astype(np.float32)
distance_km = np.where(online == 1, 0.0, rng.gamma(1.5, 4.0, n_tx))
tx_last_24h = rng.poisson(1.5 + 1.0 * online).astype(np.float32)

anomaly_type = np.array(['normal'] * n_tx, dtype=object)
anomalous = rng.choice(n_tx, int(0.012 * n_tx), replace=False)
for index, kind in zip(anomalous, rng.choice(['monto_extremo', 'rafaga', 'combinacion_atipica'], len(anomalous))):
    anomaly_type[index] = kind
    if kind == 'monto_extremo':
        log_amount[index] = mean_log_amount[category[index]] + rng.uniform(2.0, 3.0)
        hour[index] = rng.uniform(2, 5)
    elif kind == 'rafaga':
        tx_last_24h[index] = rng.integers(12, 25)
    else:  # online, lejos del domicilio y en combustible: cada valor es común, la combinación no
        category[index], online[index] = 1, 1.0
        distance_km[index] = rng.gamma(1.5, 4.0)
        log_amount[index] = rng.normal(mean_log_amount[1], 0.5)
is_anomaly = (anomaly_type != 'normal').astype(int)

features = np.column_stack([
    log_amount, np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24), online,
    np.log1p(distance_km), tx_last_24h, np.eye(8)[category]]).astype(np.float32)
feature_names = ['log_monto', 'hora_sin', 'hora_cos', 'online', 'log_distancia', 'tx_24h'] + \
                [f'cat_{name}' for name in category_names]

order = rng.permutation(n_tx)
train_idx, val_idx, test_idx = order[:18000], order[18000:24000], order[24000:]
print('variables:', features.shape[1], '| anomalías:', is_anomaly.sum())
print('por tipo:', {k: int((anomaly_type == k).sum()) for k in ['monto_extremo', 'rafaga', 'combinacion_atipica']})

**Resultado e interpretación.** El set de entrenamiento contiene anomalías sin marcar (como en la realidad: no sabemos cuáles son). Las etiquetas `is_anomaly` y `anomaly_type` se usan **solo para evaluar**, nunca para entrenar.

### Estandarizar y calcular dos baselines
Estandarizamos con media y desviación de train. El **z-score máximo** marca valores extremos en cualquier variable por separado. **Mahalanobis** considera correlaciones lineales entre variables.

In [ ]:
feature_mean = features[train_idx].mean(axis=0)
feature_std = features[train_idx].std(axis=0) + 1e-6
X = (features - feature_mean) / feature_std


def average_precision(y_true, scores):
    order = np.argsort(-scores)
    hits = y_true[order]
    precision_at_hit = np.cumsum(hits) / np.arange(1, len(hits) + 1)
    return float((precision_at_hit * hits).sum() / hits.sum())


REVIEW_CAPACITY = 0.01  # el equipo revisa el 1 % de mayor puntaje


def evaluate_scores(scores, idx):
    y = is_anomaly[idx]
    k = int(len(idx) * REVIEW_CAPACITY)
    top = np.argsort(-scores)[:k]
    result = {'precision_promedio': round(average_precision(y, scores), 4),
              'precision_capacidad': round(float(y[top].mean()), 4),
              'recall_capacidad': round(float(y[top].sum() / y.sum()), 4)}
    for kind in ['monto_extremo', 'rafaga', 'combinacion_atipica']:
        of_kind = anomaly_type[idx] == kind
        result[f'recall_{kind}'] = round(float(of_kind[top].sum() / max(of_kind.sum(), 1)), 3)
    return result


zscore_max = np.abs(X).max(axis=1)
covariance = np.cov(X[train_idx], rowvar=False) + 1e-3 * np.eye(X.shape[1])
precision_matrix = np.linalg.inv(covariance)
mahalanobis = np.einsum('ij,jk,ik->i', X, precision_matrix, X)

baseline_scores = {'zscore_max': zscore_max, 'mahalanobis': mahalanobis}
for name, scores in baseline_scores.items():
    print(f'{name:12s} validación: {evaluate_scores(scores[val_idx], val_idx)}')

**Resultado e interpretación.** Con 1,2 % de anomalías, un puntaje aleatorio tendría precisión promedio cercana a 0,012. Observa qué tipos detecta cada baseline: el z-score capta extremos univariados (monto, ráfaga) pero no tiene cómo ver la combinación atípica, porque ningún valor individual es extremo.

### Autoencoder con cuello de botella
El codificador reduce 14 variables a 4 y el decodificador las reconstruye. El cuello de botella obliga a aprender la estructura común en lugar de copiar la entrada.

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, n_features, bottleneck=4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(),
                                     nn.Linear(16, bottleneck))
        self.decoder = nn.Sequential(nn.Linear(bottleneck, 16), nn.ReLU(), nn.Linear(16, 32), nn.ReLU(),
                                     nn.Linear(32, n_features))

    def forward(self, x):
        return self.decoder(self.encoder(x))


X_t = {name: torch.tensor(X[idx], dtype=torch.float32, device=device)
       for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]}


def train_autoencoder(bottleneck=4, epochs=80, lr=2e-3, batch_size=256, patience=8):
    torch.manual_seed(SEED)
    model = AutoEncoder(X.shape[1], bottleneck).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_t['train']), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    curve, best_loss, best_state, wait = [], np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for (x,) in loader:
            loss = F.mse_loss(model(x), x)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(X_t['val']), X_t['val']).item()  # sin etiquetas: solo reconstrucción
        curve.append(val_loss)
        if val_loss < best_loss - 1e-5:
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, curve


def reconstruction_error(model, split, per_feature=False):
    model.eval()
    with torch.no_grad():
        squared = (model(X_t[split]) - X_t[split]) ** 2
    squared = squared.cpu().numpy()
    return squared if per_feature else squared.mean(axis=1)


start = time.perf_counter()
autoencoder, curve = train_autoencoder()
print(f'{len(curve)} épocas | {time.perf_counter() - start:.1f} s | MSE validación final: {min(curve):.4f}')
plt.plot(curve)
plt.xlabel('época')
plt.ylabel('MSE de reconstrucción (validación)')
plt.show()

**Resultado e interpretación.** El criterio de *early stopping* es la reconstrucción de validación, que no usa etiquetas: el procedimiento completo es no supervisado. Un MSE muy bajo no es necesariamente mejor: si el modelo reconstruye todo perfecto, también reconstruirá las anomalías.

### Comparar métodos con la capacidad de revisión
Comparamos los tres puntajes en validación y luego en test. La distribución del error muestra cuánto se separan las anomalías del resto.

In [ ]:
all_scores = {**{name: scores[val_idx] for name, scores in baseline_scores.items()},
              'autoencoder': reconstruction_error(autoencoder, 'val')}
for name, scores in all_scores.items():
    print(f'{name:12s} validación: {evaluate_scores(scores, val_idx)}')

test_error = reconstruction_error(autoencoder, 'test')
print('\nautoencoder  test:', evaluate_scores(test_error, test_idx))
print('mahalanobis  test:', evaluate_scores(mahalanobis[test_idx], test_idx))

bins = np.linspace(0, np.quantile(test_error, 0.999), 60)
plt.hist(test_error[is_anomaly[test_idx] == 0], bins=bins, alpha=0.6, density=True, label='normal')
plt.hist(test_error[is_anomaly[test_idx] == 1], bins=bins, alpha=0.6, density=True, label='anomalía')
plt.axvline(np.quantile(test_error, 1 - REVIEW_CAPACITY), color='black', linewidth=0.8, label='corte de capacidad')
plt.xlabel('error de reconstrucción')
plt.ylabel('densidad')
plt.legend()
plt.show()

**Resultado e interpretación.** Compara el *recall* por tipo, no solo la métrica global. Si el *autoencoder* recupera la combinación atípica mejor que el z-score y que Mahalanobis, es porque aprendió relaciones no lineales entre variables (online implica distancia cero). A la vez, Mahalanobis puede ser mejor en extremos de monto o ráfagas: los métodos pueden ser **complementarios**, lo que motiva la siguiente celda.

### Combinar puntajes
Si cada método detecta tipos distintos, combinarlos puede superar a ambos. Como las escalas no son comparables, promediamos **rangos**: cada transacción recibe su percentil en cada método y se ordena por el promedio.

In [ ]:
def rank_score(scores):
    ranks = np.empty(len(scores))
    ranks[np.argsort(scores)] = np.arange(len(scores))
    return ranks / (len(scores) - 1)


val_combined = (rank_score(mahalanobis[val_idx]) + rank_score(reconstruction_error(autoencoder, 'val'))) / 2
test_combined = (rank_score(mahalanobis[test_idx]) + rank_score(test_error)) / 2
print('combinado validación:', evaluate_scores(val_combined, val_idx))
print('combinado test:      ', evaluate_scores(test_combined, test_idx))

**Resultado e interpretación.** Compara con cada método por separado en validación. Un ensamble por rangos es simple, no requiere etiquetas y es fácil de explicar: "la transacción es rara según dos criterios distintos". Si mejora en validación, esa es la configuración a llevar a test; si no, conserva el método individual más simple.

### Explicar una alerta
El error por variable indica qué parte de la transacción no calza con el comportamiento aprendido. Es una explicación aproximada y útil para el analista que revisa la alerta.

In [ ]:
per_feature = reconstruction_error(autoencoder, 'test', per_feature=True)
top_alerts = np.argsort(-test_error)[:3]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for axis, alert in zip(axes, top_alerts):
    contribution = per_feature[alert] / per_feature[alert].sum()
    axis.barh(feature_names, contribution)
    axis.set_title(f'alerta {alert}: tipo real = {anomaly_type[test_idx][alert]}', fontsize=9)
    axis.set_xlabel('fracción del error')
plt.tight_layout()
plt.show()

**Resultado e interpretación.** Una alerta de ráfaga debería concentrar el error en `tx_24h`; una de combinación atípica, en `online`, `log_distancia` y la categoría. Estas explicaciones ayudan a priorizar, pero no prueban fraude: la decisión final sigue siendo humana.

## Limitaciones y conclusiones clave
Las anomalías se generaron con tipos conocidos; el fraude real cambia cuando los defraudadores se adaptan, y la proporción de anomalías en train afecta al *autoencoder*. En producción se monitorea la distribución del puntaje (deriva), se recalibra la capacidad de revisión y se incorporan las etiquetas confirmadas para pasar a modelos supervisados o semi-supervisados.

- El error de reconstrucción de un *autoencoder* es un puntaje de anomalía que aprende estructura entre variables.
- Los baselines (z-score, Mahalanobis) son obligatorios: muestran si la complejidad adicional aporta.
- Se evalúa ordenamiento dentro de la capacidad de revisión y por tipo de anomalía, no exactitud.

## Ejercicios progresivos
1. **Guiado:** entrena con `bottleneck=2` y `bottleneck=10`; compara el MSE de validación y la precisión promedio. Explica por qué un cuello de botella demasiado ancho puede empeorar la detección.
2. **Independiente:** entrena el *autoencoder* solo con transacciones normales de train (usando las etiquetas solo para filtrar) y compara con el entrenamiento contaminado. ¿Cuánto vale tener datos limpios?
3. **Desafío:** agrega un cuarto tipo de anomalía, por ejemplo restaurante a las 9 de la mañana con monto alto, sin reentrenar. Evalúa qué método lo detecta y discute cómo monitorearías anomalías nuevas en producción.

## Conexión con el siguiente laboratorio
Muchas alertas y reclamos llegan como texto libre. En el laboratorio 25 clasificamos tickets de clientes para enrutarlos automáticamente.